In [ ]:
# Vamos a obtener los diferentes accession numbers de los genomas del Acinetobacter complex
# Para ello usaremos Biopython - módulo Entrez, concretamente esearch

from Bio import Entrez
import os
import urllib.request
import gzip
import shutil

Entrez.email = "X@gmail.com" 

# Para lo que queremos usaremos 2 módulos distintos de Entrez: 
# esearch - "dame los IDs que coinciden con mi búsqueda"
# esummary - "dame los detalles de este ID concreto"

# Primero definimos las especies de nuestra búsqueda. Así es fácil añadir o quitar especies en el futuro

os.makedirs("Acinetobacter_complex_genomas", exist_ok=True)

especies = [ # Creamos una lista
    "Acinetobacter baumannii",
    "Acinetobacter seifertii",
    "Acinetobacter calcoaceticus",
    "Acinetobacter lactucae",
    "Acinetobacter nosocomialis",
    "Acinetobacter pittii"
]

for especie in especies:
    query = f'"{especie}"[Organism] AND "complete genome"[Assembly level]'

    stream = Entrez.esearch (db="assembly",
                         term=query,
                         retmax=2)
    record= Entrez.read(stream)
    stream.close()
    # Para que cada especie puede tener más de un genoma, necesitamos hacer un bucle dentro de este bucle

    for assembly_id in record['IdList']:
        summary_stream = Entrez.esummary(db="assembly",id=assembly_id)
        summary = Entrez.read (summary_stream, validate=False)
        summary_stream.close ()

        doc = summary ["DocumentSummarySet"]["DocumentSummary"][0]
        ftp_path = doc.get("FtpPath_RefSeq") or doc.get("FtpPatch_Genbank") # FTP es un sistema para transferir archivos por internet. 
        # NCBI guarda todos sus genomas en un servidor FTP, y cada ensamblaje tiene una ruta (como una dirección) donde están sus archivos. 
        # Esa ruta es lo que guardamos en ftp_path

        if ftp_path and ftp_path != "na": # Quiere decir: "Si la variable no está vacía y no es igual a na", entonces procede con lo siguiente

            nombre_archivo = ftp_path.split("/")[-1] # Esta línea es la que usaremos para ponerle el nombre a nuestro archivo, teniendo en cuenta que un ftp path se ve tal que:
            # https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/GCF_000001405.40_GRCh38.p14
            # split lo trocea por cada " / " y [-1] coge solo el último cacho, que coincide con el nombre del ensamblaje!!

            url = f"{ftp_path}/{nombre_archivo}_genomic.fna.gz".replace("ftp://", "https://")
            # Construye la URL completa del archivo a descargar, añadiendo al final _genomic.fna.gz, que es como NCBI nombra siempre los archivos FASTA.

            destino_gz = f"Acinetobacter_complex_genomas/{nombre_archivo}.fna.gz"
            # Define dónde y con qué nombre se guardará el archivo descargado
            # .gz significa que está comprimido!! Nostros queremos directamente el .fna

            destino_fna = f"Acinetobacter_complex_genomas/{nombre_archivo}.fna"

            urllib.request.urlretrieve(url, destino_gz)
            # Esta es la línea que REALMENTE descarga el archivo. Le decimos de dónde descargarlo (url) y dónde guardarlo (destino_gz)

            # DESCOMPRIMIR Y BORRAR LOS .GZ

            with gzip.open(destino_gz, "rb") as f_in: # Con gzip abre el archivo y léelo (rb - read binary, el método que se necesita con comprimidos)
                with open(destino_fna, "wb") as f_out: # Con comando open normal creamos un archivo .fna donde le diremos que escriba (wb - write)
                    shutil.copyfileobj(f_in, f_out) # Esta es la línea que realmente descomprime. Copia el contenido de f_in (el .gz) hacia f_out (el .fna), descomprimiéndolo por el camino. 
            os.remove(destino_gz)

# Al buscar las especies en la base de datos de genome vemos que no siempre coinciden las cepas con el 1er y 2do resultado que nos salen en las búsquedas.
# Esto es porque cuando buscas en la web, NCBI te muestra los resultados con su propia interfaz visual que puede aplicar filtros y ordenaciones adicionales por debajo 
# que no se ven. En cambio, cuando hacemos la búsqueda con esearch desde Python, accedemos directamente a la base de datos sin esos filtros visuales, 
# así que el orden puede ser diferente.


In [ ]:
print (summary)

{'DocumentSummarySet': DictElement({'DocumentSummary': [DictElement({'RsUid': '', 'GbUid': '82292888', 'AssemblyAccession': 'GCA_055475395.1', 'LastMajorReleaseAccession': 'GCA_055475395.1', 'LatestAccession': '', 'ChainId': '55475395', 'AssemblyName': 'ASM5547539v1', 'UCSCName': '', 'EnsemblName': '', 'Taxid': '48296', 'Organism': 'Acinetobacter pittii (g-proteobacteria)', 'SpeciesTaxid': '48296', 'SpeciesName': 'Acinetobacter pittii', 'AssemblyType': 'haploid', 'AssemblyStatus': 'Complete Genome', 'AssemblyStatusSort': '1', 'WGS': '', 'GB_BioProjects': [{'BioprojectAccn': 'PRJNA1005445', 'BioprojectId': '1005445'}], 'GB_Projects': [], 'RS_BioProjects': [], 'RS_Projects': [], 'BioSampleAccn': 'SAMN36987370', 'BioSampleId': '36987370', 'Biosource': {'InfraspeciesList': [], 'Sex': '', 'Isolate': ''}, 'Coverage': '1.1', 'PartialGenomeRepresentation': 'false', 'Primary': '82292898', 'AssemblyDescription': '', 'ReleaseLevel': 'Major', 'ReleaseType': 'Major', 'AsmReleaseDate_GenBank': '2026